# 00 -- Setup

Brings up a working Colab environment from scratch. Gate (plan Step 0): this notebook should run cleanly top to bottom (Runtime -> Restart and run all) and confirm GPU, Drive, Stockfish, and the HF model all work.

Fill in `REPO_URL` and `HF_MODEL_ID` below before running. Add your Hugging Face read token to Colab Secrets as `HF_TOKEN` first (key icon in the left sidebar).

In [ ]:
REPO_URL = "https://github.com/<you>/Chess_Bot.git"  # TODO: fill in once you've created + pushed this repo
HF_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # TODO: confirm exact checkpoint on the Hub (plan: smallest current Qwen, ~0.5-1B, Apache-2.0)

## Clone the repo

In [ ]:
import os

if not os.path.exists("Chess_Bot"):
    !git clone {REPO_URL}
%cd Chess_Bot

## Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Mount Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/chess-ai'  # adjust if you want a different Drive path
os.makedirs(DRIVE_ROOT, exist_ok=True)

## GPU + CPU check

Log these every run -- the plan's fairness rule is that every *reported* run must use the same GPU type, and Stockfish labeling/eval timing (Steps 2-3) depends on CPU core count.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CPU cores:", os.cpu_count())

## Stockfish

Downloads the official Linux x86-64 **universal** binary (Stockfish 19+ auto-detects CPU features -- there's no separate avx2 asset anymore) from the GitHub releases page. Fallback if this fails: `!apt-get install -y stockfish` (an older version, but works).

In [ ]:
!wget -q -O stockfish.tar.gz https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-linux-x86-64-universal.tar.gz
!tar -xzf stockfish.tar.gz stockfish/stockfish-linux-x86-64-universal
!chmod +x stockfish/stockfish-linux-x86-64-universal

import chess
import chess.engine

engine = chess.engine.SimpleEngine.popen_uci("./stockfish/stockfish-linux-x86-64-universal")
board = chess.Board()
info = engine.analyse(board, chess.engine.Limit(depth=10))
print("Stockfish eval from startpos:", info["score"])
engine.quit()

## Hugging Face model

Requires `HF_TOKEN` in Colab Secrets (a read token from huggingface.co/settings/tokens).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(HF_MODEL_ID)

inputs = tokenizer("The Sicilian Defense is", return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0]))

## Gate

If every cell above ran without error on a fresh runtime, Step 0 is done. Move to Step 1 (`engine/`).